In [1]:
# 전체 패키지 설치
!pip install -qU langchain langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install -q faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 491.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 23.7 MB/s eta 0:00:00


In [ ]:
# legacy 안정화 호환 버전
# 0.3.x 시리즈 내에서 안정성이 확인된 버전들로 설치
!pip install -q \
    langchain>=0.3.0 \
    langchain-core>=0.3.0 \
    langchain-community>=0.3.0 \
    langchain-openai>=0.3.0 \
    langchain-text-splitters>=0.3.0 \
    faiss-cpu \
    tiktoken \
    python-dotenv

In [2]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

프롬프트 템플릿 적용
  - langchain_core의 prompts 모듈에서 ChatPromptTemplate 클래스를 사용하여 천문학 전문가로서의 답변 형식의 프롬프트 템플릿 생성

In [3]:
# 모델과 연결하여 실행하기

# 1. 필수 라이브러리 임포트
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

# *주의: 환경변수에 OPENAI_API_KEY가 설정되어 있어야 합니다.
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# 2. 프롬프트 템플릿 정의 (prompt 객체 생성)
prompt = ChatPromptTemplate.from_template("You are an expert in astronomy, Answer the question.<Question> : {input}")


# 3. 모델 설정
model = ChatOpenAI(model="gpt-4o", temperature= 0.7)


# 4. 체인 구성 (LCEL 방식)
chain = prompt | model | StrOutputParser()

# 5. 실행
result = chain.invoke({"input": "블랙홀의 사건의 지평선이 무엇인가요?"})
print(result)

블랙홀의 사건의 지평선(event horizon)은 블랙홀 주변에서 중력이 너무 강하여 빛조차 빠져나갈 수 없는 경계입니다. 이 지평선을 넘어서면 물체는 블랙홀로 빨려 들어가게 되며, 외부 관측자가 그 물체를 볼 수 없게 됩니다. 사건의 지평선은 블랙홀의 크기를 정의하는 중요한 요소이며, 이 경계를 넘어서면 블랙홀 내부의 상태나 구조에 대한 정보는 이론적으로 외부로 전달될 수 없습니다. 사건의 지평선은 일반 상대성이론에 의해 예측된 개념으로, 블랙홀의 특성 중 하나입니다.


In [4]:
# 전문가의 팁: 프롬프트 최적화
# from_template으로 단순하게 작성하셨지만, 조금 더 정교한 답변을 원하신다면
# SystemMessage를 명시적으로 분리

from langchain_core.prompts import ChatPromptTemplate

# 1. 프롬프트 템플릿 생성
# 시스템 메시지(System Message)로 페르소나를 정의합니다.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 30년 경력의 천문학 전문가입니다. 복잡한 우주 현상을 일반인도 이해하기 쉽게 \
    비유를 들어 설명하며, 항상 과학적 사실에 기반하여 정확하게 답변하세요."),
    ("human", "질문 : {question}"),
    ("ai", "천문학자의 답변 : ")
])


# 2. 템플릿 테스트 (예시 데이터 적용)
formatted_prompt = prompt_template.format(question = "블랙홀은 어떻게 생성되나요?")


print(formatted_prompt)

System: 당신은 30년 경력의 천문학 전문가입니다. 복잡한 우주 현상을 일반인도 이해하기 쉽게     비유를 들어 설명하며, 항상 과학적 사실에 기반하여 정확하게 답변하세요.
Human: 질문 : 블랙홀은 어떻게 생성되나요?
AI: 천문학자의 답변 : 


In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = 'gpt-4o-mini')

# LCEL(LangChain Expression Language)
# chain = prompt_template | llm        # | : 서로 다른 구성 요소를 연결

chain = prompt_template | llm | StrOutputParser()    # AIMessage 객체에서 텍스트 부분만 추출하여 반환

response = chain.invoke({'question' : '지구의 윤년에 주기는?'})
print(response)

지구의 윤년 주기는 대략 4년입니다. 일반적으로, 1년은 365일이지만, 태양이 지구를 한 바퀴 도는 데 걸리는 정확한 시간은 약 365일 5시간 48분 45초입니다. 이 초과된 시간이 쌓이게 되면, 결국에는 1년이 더 필요하게 됩니다.

이렇게 초과된 시간을 보충하기 위해 4년마다 1일을 추가하여 2월에 29일을 선물하는 것이 바로 윤년입니다. 따라서, 윤년은 4로 나누어 떨어지는 해가 되는 것이지만, 특정 규칙도 있습니다. 예를 들어, 100으로 나누어 떨어지는 해는 윤년이 아니지만, 400으로 나누어 떨어지는 해는 윤년이 됩니다. 

이를 일반인에게 비유하자면, 4년마다 한 번씩 생일 파티를 크게 열어 친구들을 초대하는 것과 비슷합니다. 매년 생일이 있지만, 4년에 한 번 특별한 날이 더해져서 더욱 큰 축하를 하는 셈이죠.


In [12]:
result = chain.invoke({'question' : '지구의 윤년에 주기는?'})
result.content

# StrOutputParser() 가 없는 경우에는 객체를 반환하기 때문에 텍스트를 출력하기 위해서는
# result.content의 형식이 필요

AttributeError: 'TextAccessor' object has no attribute 'content'

LCEL
  - 기본 LLM 체인
  - 프롬프트, LLM, 문자열 출력 파서(StrOutputParser)를 연결하여 체인을 만들 수 있다

LCEL (Language Chain Execution Layer)

정의:

LangChain에서 말하는 개념으로, LLM과 외부 도구(API, DB 등)를 연결해 실제 작업을 수행하도록 체인(chain)을 구성하는 레이어

쉽게 말하면 LLM을 단순 생성기 → 실제 업무 수행기로 바꿔주는 매개층

역할:

프롬프트 관리

사용자가 입력한 자연어 → 모델이 이해할 수 있는 형태로 변환

체인 연결

LLM → 도구 → 데이터베이스 → LLM 순으로 연결 가능

출력 파싱 및 후처리

모델 출력 정리, JSON 변환, 요약 등

In [13]:

# 라이브러리 임포트
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 간단한 프롬프트 템플릿 생성
template = "{country}의 수도는 어디인가요?"
prompt_template = PromptTemplate.from_template(template)


# 프롬프트 테스트
print(prompt_template.format(country="대한민국"))
print(prompt_template.format(country="미국"))




대한민국의 수도는 어디인가요?
미국의 수도는 어디인가요?


In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# prompt + model + output parser
prompt = ChatPromptTemplate.from_template('You are an expert in astronomy, Answer the question. <Question> : {input}')


llm = ChatOpenAI(model = "gpt-4o-mini")


output_parser = StrOutputParser()


# LCEL chaining
chain = prompt | llm | output_parser


# chain 호출
result = chain.invoke({'input' : '지구의 윤년에 주기는?'})

# StrOutputParser()를 사용하면 AIMessage 객체가 아닌
# 순수문자열만 반환되기 때문에 content속성이 없다

print(result)

지구의 윤년 주기는 4년에 한 번입니다. 그러나 100년 단위에서는 윤년이 아니라는 규칙이 있으며, 400년 단위에서는 다시 윤년으로 간주됩니다. 즉, 일반적인 규칙은 다음과 같습니다:

1. 연도가 4로 나누어 떨어지면 윤년입니다.
2. 그러나 연도가 100으로 나누어 떨어지면 윤년이 아닙니다.
3. 다시 말해, 연도가 400으로 나누어 떨어지면 윤년입니다.

이런 규칙 덕분에 지구의 공전 주기와 날짜 시스템이 잘 맞춰지게 됩니다.


템플릿 내용 변경하여 테스트

In [15]:
# ChatOpenAI 모델 초기화
model = ChatOpenAI(
    model = "gpt-4o-mini",
    max_tokens = 2048,
    temperature= 0.1
)



# 출력 파서 초기화
output_parser = StrOutputParser()


# 체인 생성: 프롬프트 -> 모델 -> 출력 파서
prompt = PromptTemplate.from_template("{topic}에 대해 쉽게 설명해 주세요.")

chain = prompt | model | output_parser

# 입력 딕셔너리
input_data ={"topic" : "인공지능 모델의 학습 원리"}


# 체인 실행
result = chain.invoke(input_data)
print("=== 결과 ===")
print(result)

# 영어 회화 예제
template_eng = """
당신은 영어를 가르치는 10년차 영어 선생님입니다.
상황에 [FORMAT]에 영어 회화를 작성해 주세요.

상황:
{question}

FORMAT:
- 영어 회화:
- 한글 해석:
"""
prompt_eng = PromptTemplate.from_template(template_eng)

model_eng = ChatOpenAI(model = "gpt-4-turbo")

chain_eng = prompt_eng | model_eng | output_parser


# 질문 실행
input_question = {"question": "저는 식당에 가서 음식을 주문하고 싶어요"}
result_eng = chain_eng.invoke(input_question)
print("=== 영어 회화 예제 ===")
print(result_eng)

# 또 다른 질문
input_question2 = {"question": "미국에서 피자 주문"}
result_eng2 = chain_eng.invoke(input_question2)
print("=== 미국 피자 주문 예제 ===")
print(result_eng2)

=== 결과 ===
인공지능 모델의 학습 원리는 크게 두 가지 단계로 나눌 수 있습니다: **훈련(Training)**과 **테스트(Test)**입니다. 이를 쉽게 설명해 보겠습니다.

1. **훈련(Training)**:
   - **데이터 수집**: 먼저, 모델이 학습할 데이터를 수집합니다. 이 데이터는 이미지, 텍스트, 숫자 등 다양한 형태일 수 있습니다.
   - **모델 선택**: 학습할 모델을 선택합니다. 예를 들어, 이미지 인식에는 CNN(합성곱 신경망)을, 자연어 처리에는 RNN(순환 신경망)이나 Transformer를 사용할 수 있습니다.
   - **학습**: 모델은 주어진 데이터를 바탕으로 패턴을 학습합니다. 이 과정에서 모델은 입력 데이터와 정답(라벨)을 비교하여 오차를 계산하고, 이 오차를 줄이기 위해 가중치를 조정합니다. 이 과정을 여러 번 반복하면서 모델이 점점 더 정확해집니다.

2. **테스트(Test)**:
   - **검증**: 훈련이 끝난 후, 모델의 성능을 평가하기 위해 새로운 데이터(테스트 데이터)를 사용합니다. 이 데이터는 모델이 훈련할 때 사용하지 않았던 데이터로, 모델이 얼마나 잘 일반화되었는지를 확인하는 데 사용됩니다.
   - **결과 분석**: 모델이 테스트 데이터에 대해 얼마나 정확하게 예측하는지를 분석합니다. 이를 통해 모델의 성능을 평가하고, 필요하다면 모델을 개선하는 작업을 진행합니다.

이러한 과정을 통해 인공지능 모델은 주어진 문제를 해결할 수 있는 능력을 갖추게 됩니다. 쉽게 말해, 많은 데이터를 통해 '배우고', 그 배운 내용을 바탕으로 새로운 상황에서도 '예측'할 수 있게 되는 것입니다.
=== 영어 회화 예제 ===
영어 회화:  
"Hello, I'd like to have a table for one, please. Could I see the menu? I think I'll go for the grilled chicken with a side of mashed potatoes, and 

문맥만을 기반으로 답변을 생성하는 체인

이 코드는 LangChain 라이브러리를 사용하여, 간단한 문장을 벡터화(임베딩)한 후 FAISS라는 벡터 인덱스에 저장합니다. 사용자가 질문을 입력하면, 저장된 벡터 데이터에서 관련된 문맥을 검색해 그 문맥만을 기반으로 답변을 생성하는 체인을 만듭니다.

- FAISS 벡터 저장소에 텍스트 임베딩 저장

- 벡터 검색기(retriever)를 만들어 관련 문맥 추출

- 문맥과 질문을 특정 프롬프트 템플릿에 넣음

- OpenAI GPT-4o-mini 모델에 프롬프트 전달해 답변 생성

- 결과를 문자열로 파싱하여 출력

- 마지막으로 이 전체 체인의 노드와 연결 관계를 그래프로 시각화

In [16]:
# 필요한 패키지 설치
!pip install -qU faiss-cpu tiktoken grandalf langchain langchain_openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.3 MB/s eta 0:00:00


In [ ]:
# 강제 업데이트 및 설치 -> 세션 다시 시작
!pip install --upgrade grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.6 MB/s eta 0:00:00


In [17]:
# 문맥만을 기반으로 답변을 생성하는 RAG(검색 증강 생성) 시스템은 현대적인 AI 어시스턴트의 핵심입니다.
# 설명해주신 단계들을 LangChain의 LCEL 구조로 구현하면 매우 견고한 시스템
# 표준적이고 유지보수가 중요한 경우: 첫 번째 코드의 RunnablePassthrough 방식을 권장합니다.
# 코드가 더 간결하고 LangChain의 철학에 가장 부합합니다.


from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. FAISS 벡터 저장소 구축
texts = ["천문학은 우주에 관한 과학입니다.", "지구의 윤년은 4년마다 돌아옵니다."]

vectorstore = FAISS.from_texts(texts, embedding= OpenAIEmbeddings())

retriever = vectorstore.as_retriever()


# 2. 프롬프트 템플릿: 오직 주어진 {context}만 활용하도록 설정
template = """다음 문맥만을 사용하여 질문에 답변하세요 :

{context}

질문 : {question}

"""


prompt = ChatPromptTemplate.from_template(template)


# 3. 모델 및 파서
model = ChatOpenAI(model = "gpt-4o-mini")

output_parser = StrOutputParser()


# 4. RAG 체인 구축
# RunnablePassthrough를 사용해 context와 question을 프롬프트에 주입
chain = (
    {"context" : retriever, "question" : RunnablePassthrough()}
    | prompt
    | model
    | output_parser
)

# 5. 실행
result = chain.invoke("지구의 윤년 주기는?")
print(result)

# 6. 체인 구조 시각화 (도식화)
# .get_graph().print_ascii()를 사용하면 체인의 흐름을 터미널에서 볼 수 있습니다.

print(chain.get_graph().print_ascii())



/tmp/ipykernel_1498/3457521366.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


지구의 윤년 주기는 4년마다 돌아옵니다.
           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                         

In [18]:
# 필요한 라이브러리 임포트
# 수정된 import 경로
# 입력값 가공이 필요한 경우: 두 번째 코드의 RunnableLambda 방식을 선택하세요.
# 단순히 통과시키는 것을 넘어 데이터 흐름 중간에 특정 로직을 끼워 넣기 쉽습니다.

from langchain_core.prompts import ChatPromptTemplate  # langchain.prompts 대신 사용
from langchain_community.vectorstores import FAISS      # langchain.vectorstores 대신 사용
from langchain_core.output_parsers import StrOutputParser   # 출력 결과를 문자열로 변환하는 파서
from langchain_core.runnables import RunnableLambda         # 람다 함수 실행 래퍼
from langchain_openai import ChatOpenAI, OpenAIEmbeddings   # OpenAI 챗 모델과 임베딩 생성용

import os
# OpenAI API 키를 환경 변수에 설정 (본인 키로 변경 필요)
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")


# 텍스트 데이터를 벡터로 변환하여 FAISS 벡터 인덱스 생성
vectorstore = FAISS.from_texts(
   ["천문학은 우주에 관한 과학입니다.", "지구의 윤년은 4년마다 돌아옵니다."],
   embedding= OpenAIEmbeddings(),
)


# 생성한 벡터 저장소를 기반으로 검색기(retriever) 생성
# 질문에 대해 관련 텍스트를 벡터 유사도로 찾아줌
retriever = vectorstore.as_retriever()



# 프롬프트 템플릿 정의
# 검색된 문맥(context)만 참고해서 질문에 답변하라는 지시문 포함
template = """ 다음 문맥만을 사용하여 질문에 답변하세요.
{context}

Question : {question}
"""

# 위 문자열 템플릿을 LangChain 프롬프트 객체로 변환
prompt = ChatPromptTemplate.from_template(template)


# OpenAI 챗 모델 인스턴스 생성
# 여기서는 경량화된 "gpt-4o-mini" 모델 사용
model = ChatOpenAI(model = "gpt-4o-mini")



# passthrough는 입력값을 그대로 출력하는 람다 함수
# 질문을 별도 변환 없이 그대로 체인에 전달할 때 사용
passthrough = RunnableLambda(lambda x:x)


# LangChain 체인 구성
# - retriever가 context(문맥)를 반환
# - passthrough가 question(질문)을 그대로 전달
# - prompt가 context와 question을 템플릿에 넣어 프롬프트 생성
# - model이 프롬프트를 받아 답변 생성
# - StrOutputParser가 답변을 문자열로 파싱
chain = (
    {"context" : retriever, "question" : passthrough}
    | prompt
    | model
    | StrOutputParser()
)


# 체인 내부의 그래프 노드(단계) 출력
print("Nodes:")
print(chain.get_graph().nodes)


# 체인 그래프 내 엣지(연결 관계) 출력
print("Edges:")
print(chain.get_graph().edges)


# ASCII 아트 형태로 체인 전체 그래프 구조 시각화 출력
print("Graph:")
chain.get_graph().print_ascii()



Nodes:
{'d552cb0cd80f4885bfc86607ec849b8a': Node(id='d552cb0cd80f4885bfc86607ec849b8a', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None), '3c3e1d83a5f34f5790b571a8c72239ba': Node(id='3c3e1d83a5f34f5790b571a8c72239ba', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None), '17f3aabec8de4b5da4b448415bd016c5': Node(id='17f3aabec8de4b5da4b448415bd016c5', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d2b3456b860>, search_kwargs={}), metadata=None), 'cc8295461d924b959cf5310663930c59': Node(id='cc8295461d924b959cf5310663930c59', name='Lambda', data=RunnableLambda(lambda x: x), metadata=None), 'e455b6de6c7947819fc9bd7ec7e5fa33': Node(id='e455b6de6c7947819fc9bd7ec7e5fa33', name='ChatPromptTemplate', dat

LangChain Expression Language에서 라우팅을 수행하는 방법

LCEL로 라우팅을 수동으로 구현한 구조입니다
- 구성 요소	      : 설명
- `	              : ` 연산자
- PromptTemplate	: 입력을 원하는 형식으로 구성
- ChatOpenAI	    : LLM 모델 호출
- StrOutputParser	: 응답을 문자열로 변환
- chain.invoke()	: 실행 트리거
- 라우팅 로직	분류 결과에 따라 적절한 체인 호출

In [ ]:
# 실행 불필요
!pip install -U -q langchain langchain-core langchain-community langchain-openai

In [19]:
# legacy
# LCEL에서 라우팅(Routing)은 질문의 의도나 주제에 따라
# 서로 다른 체인으로 데이터를 전달하는 매우 강력한 기능입니다.
# 이를 구현하기 위해 주로 RunnableBranch를 사용하거나,
# 사용자의 질문을 분석해 다음 단계의 체인을 선택하는 '분류기(Classifier)'를 앞에 배치


# 필요한 라이브러리 임포트
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 사용자 질문을 분류하는 프롬프트 템플릿
prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)

# 체인 생성
chain = (
    prompt| ChatOpenAI(model = "gpt-4") | StrOutputParser()
)



# 질문을 입력하여 체인을 호출
response1 = chain.invoke({"question": "2+2 는 무엇인가요?"})
response2 = chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})
response3 = chain.invoke({"question": "Google은 어떤 회사인가요?"})

print(response1)  # 출력: 수학
print(response2)  # 출력: 과학
print(response3)  # 출력: 기타

# 수학 관련 질문을 처리하는 체인
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4")
)

# 과학 관련 질문을 처리하는 체인
science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4")
)

# 기타 관련 질문을 처리하는 체인
general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4")
)

# 각 체인을 호출하여 질문을 처리
math_response = math_chain.invoke({"question": "2+2 는 무엇인가요?"})
science_response = science_chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})
general_response = general_chain.invoke({"question": "Google은 어떤 회사인가요?"})

print(math_response)  # 예시: 깨봉선생님께서 말씀하시기를..
print(science_response)  # 예시: 아이작 뉴턴 선생님께서 말씀하시기를..
print(general_response)  # 예시: Google은 검색 엔진을 제공하는 회사입니다.


수학
과학
기타
content='깨봉선생님께서 말씀하시기를, 2+2는 4입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 70, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-Dyyq7yaLZHgYRwoNceAGJoZIIlSJ5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f3c77-eace-74b0-8d04-664998813bd4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 70, 'output_tokens': 33, 'total_tokens': 103, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
content='아이작 뉴턴 선생님께서 말씀하시기를, 작용 반작용의 법칙은 어떤 물체에 힘이 작용하면 그 힘과 같은 크기이고 반대 방향으로 힘이 그 물체에게 다시 작용한다는 법칙입니다. 이것은 그의 제3법칙으로 알려져 있으며 순간적인 힘의 

In [21]:
# 개선코드
# LCEL의 파이프라인 구조를 활용하여 입력부터 최종 답변까지의 흐름을 효율적으로 제어
# 자동 라우팅 시스템으로 확장하기
# 위의 코드는 체인을 각각 따로 호출하고 계시지만,
# 아래와 같이 RunnableLambda를 사용하면
# 질문 하나만 던져도 알아서 분류하고 답변까지 도출하는 단일 체인

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

# API 키 설정
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

# 1. 분류기 체인 (어떤 주제인지 판별)
classification_prompt = PromptTemplate.from_template(
    """주어진 질문을 '수학', '과학', '기타' 중 하나로만 분류하세요. 다른 말은 하지 마세요.
    질문 : {question}
    분류 : """
)

classifier_chain = classification_prompt | ChatOpenAI(model = "gpt-4o-mini") | StrOutputParser()

# 2. 전문 답변 체인 정의
math_chain = PromptTemplate.from_template("당신은 수학 전문가입니다. '깨봉선생님께서 말씀하시기를..'으로 시작하여 설명하세요.\n질문: {question}") | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()
science_chain = PromptTemplate.from_template("당신은 과학 전문가입니다. '아이작 뉴턴 선생님께서 말씀하시기를..'으로 시작하여 설명하세요.\n질문: {question}") | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()
general_chain = PromptTemplate.from_template("당신은 친절한 도우미입니다. 간결하게 답변하세요.\n질문: {question}") | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

# 3. 라우팅 로직 (분류 결과에 따라 체인 선택)
def route(info):
    topic = info["topic"].strip()
    if "수학" in topic:
        return math_chain
    elif "과학" in topic:
        return science_chain
    else:
        return general_chain

# 4. 통합 라우팅 체인 구축
# RunnablePassthrough를 사용해 입력값을 유지하고 분류 결과와 함께 라우터로 전달
full_chain = (
    {"topic" : classifier_chain, "question" : RunnablePassthrough()}
    | RunnableLambda(route)
)



# 5. 실행
print("=== 답변 결과 ===")
print(full_chain.invoke("2+2는 무엇인가요?"))
print(full_chain.invoke("중력이란 무엇인가요?"))
print(full_chain.invoke("파리 날씨 어때?"))

=== 답변 결과 ===
깨봉선생님께서 말씀하시기를, "2+2는 무엇인가요?"라는 질문은 아주 간단하지만 중요한 수학적 개념을 포함하고 있습니다. 

우선, 숫자 2는 두 개의 단위를 의미합니다. 우리가 2개를 가지고 있는 상태에서 다시 2개를 더하면, 전체의 개수는 어떻게 될까요? 

이렇게 생각해보면, 첫 번째 2와 두 번째 2를 합치면 총 4가 됩니다. 즉, 

2 + 2 = 4입니다.

이것은 기본적인 덧셈의 원리이며, 수학의 기초를 이루는 중요한 부분입니다. 덧셈은 두 개 이상의 수를 합치는 과정으로, 이를 통해 우리는 더욱 복잡한 형태의 수학 문제도 해결할 수 있습니다. 이러한 간단한 개념을 확고히 이해하는 것이 수학의 큰 그림을 이해하는 데에 큰 도움이 됩니다.
아이작 뉴턴 선생님께서 말씀하시기를, 중력은 두 물체 사이에 작용하는 끌어당김의 힘이라고 하셨습니다. 이 힘은 물체의 질량이 클수록, 그리고 두 물체 사이의 거리가 가까울수록 강해집니다. 뉴턴은 이러한 중력의 법칙을 수치적으로 표현할 수 있는 수식, 즉 만유인력의 법칙을 발견하였고, 이를 통해 지구가 떨어지는 사과를 끌어당기는 힘과 같은 원리를 설명할 수 있었습니다.

중력은 우리가 지구에서 느끼는 힘으로, 모든 물체를 지구 중심으로 끌어당깁니다. 이는 우리가 땅에 서 있는 이유이기도 하며, 지구와 달, 태양과 행성들 간의 궤도를 결정짓는 힘입니다. 뉴턴은 이 중력을 통해 천체의 운동을 설명하고, 나중에 아인슈타인이 제안한 일반 상대성이론으로 이어지는 중력에 대한 더 깊은 이해를 제공하는 기초를 마련했습니다.

결론적으로, 중력은 물체가 서로 끌어당기게 하는 힘이며, 우주에서의 모든 천체의 운동과 상호작용에 중요한 역할을 하는 자연의 기본적인 힘 중 하나입니다.
현재 파리의 날씨는 확인할 수 없습니다. 최신 날씨 정보를 검색해 보세요.


In [ ]:
# 전체 패키지 설치
!pip install -q langchain>=0.3.0 langchain-openai langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install -q faiss-cpu tiktoken python-dotenv

# 라이브러리 설치 (중복 설치 방지를 위해 -q 옵션 사용)
!pip install -q langchain langchain-openai tiktoken

# Chroma 벡터스토어를 사용할 때 필요한 패키지
!pip install -q langchain_chroma

사용자의 질문을 자동으로 분류하고, 그에 맞는 전문가 스타일로 답변을 생성하는 LangChain 기반 OpenAI 응답 시스템

복합 체인(Composite Chain)이란?

  - 여러 개의 LangChain 체인(Chain)을 조합해서, 입력에 따라 동적으로 분기하거나 결합된 흐름으로 처리하는 구조입니다.

이 코드의 경우:

  - 분류 체인(classification_chain)
→ 입력 질문을 분석해서 "수학", "과학", "기타" 중 하나로 분류함

  - 세 가지 전문 체인(math_chain, science_chain, general_chain)
→ 각각 특정 톤과 스타일로 답변 생성

  - 라우팅 체인(router_chain)
→ 분류 결과에 따라 적절한 체인을 선택하고 실행함

RunnableLambda로 동적 분기

In [23]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

# OpenAI API 키 설정
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")


# 1. 질문 분류 체인 (수학 / 과학 / 기타)
classification_prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요.
    한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)

classification_chain = (
    classification_prompt
    | ChatOpenAI(model = "gpt-4o", temperature= 0)
    | StrOutputParser()
)



# 2. 수학 체인
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4o", temperature=0)
    | StrOutputParser()
)

# 3. 과학 체인
science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4o", temperature=0)
    | StrOutputParser()
)

# 4. 기타 체인
general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4o", temperature=0)
    | StrOutputParser()
)

# 5. 라우터 함수: 분류 결과에 따라 적절한 체인 선택
def router(input: dict):
    category = classification_chain.invoke({"question": input["question"]}).strip()
    if category == "수학":
        return math_chain
    elif category == "과학":
        return science_chain
    else:
        return general_chain

# 6. 최종 라우팅 체인 (RunnableLambda로 동적 분기) -> 실행 시점에서 체인을 결정
router_chain = RunnableLambda(lambda input : router(input).invoke(input))


# 7. 테스트 실행
questions = [
    "2+2는 무엇인가요?",
    "작용 반작용 법칙은 무엇인가요?",
    "Google은 어떤 회사인가요?",
]

for q in questions:
    print(f"\n질문: {q}")
    answer = router_chain.invoke({"question": q})
    print(f"답변: {answer}")



질문: 2+2는 무엇인가요?
답변: 깨봉선생님께서 말씀하시기를, 2+2는 4입니다.

질문: 작용 반작용 법칙은 무엇인가요?
답변: 아이작 뉴턴 선생님께서 말씀하시기를, 작용 반작용 법칙은 뉴턴의 제3법칙으로 알려져 있으며, 이는 모든 작용에는 크기가 같고 방향이 반대인 반작용이 있다는 원리입니다. 즉, 어떤 물체가 다른 물체에 힘을 가하면, 두 번째 물체도 첫 번째 물체에 동일한 크기의 반대 방향의 힘을 가한다는 것입니다. 이 법칙은 물리학에서 힘의 상호작용을 이해하는 데 중요한 역할을 합니다.

질문: Google은 어떤 회사인가요?
답변: Google은 주로 인터넷 검색 엔진으로 잘 알려진 다국적 기술 회사입니다. 검색 서비스 외에도 온라인 광고, 클라우드 컴퓨팅, 소프트웨어 및 하드웨어 제품 등을 제공합니다. Google은 알파벳(Alphabet Inc.)의 자회사입니다.


In [ ]:
import os

# OpenAI API 키를 환경 변수에 설정합니다.
# 실제 API 키는 보안상 직접 코드에 넣지 않고 환경 변수나 secrets manager를 사용하는 것이 좋습니다.
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

In [ ]:
# 전체 패키지 설치
!pip install langchain>=0.3.0 langchain-openai langchain-core langchain-community langchain-text-splitters

# 추가 유용한 패키지들
!pip install faiss-cpu tiktoken python-dotenv

# 라이브러리 설치 (중복 설치 방지를 위해 -q 옵션 사용)
!pip install -q langchain langchain-openai tiktoken

# Chroma 벡터스토어를 사용할 때 필요한 패키지
!pip install langchain_chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 45.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# LangChain 관련 라이브러리 임포트
# 1. 템플릿 관련 모듈 (경로 수정)
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
# FewShotPromptTemplate은 이제 langchain_core.prompts에서 직접 가져옵니다.
from langchain_core.prompts import FewShotPromptTemplate

# 2. 콜백 및 기타 모듈
from langchain_core.prompts import BaseCallbackHandler

# StreamingStdOutCallbackHandler는 langchain_community에서 이동했습니다.
# 기존의 langchain_community 경로 대신 아래 경로를 사용하세요
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

LangChain의 언어 모델(Model)
  - LLM과 Chat Model 클래스는 각각 다른 형태의 입력과 출력을 다루는 언어 모델을 나타낸다
  - LLM : 단일 요청에 대한 복잡한 출력을 생성
    1. 기능 : LLM 클래스는 텍스트 문자열을 입력받아 처리한 후, 텍스트 문자열을 반환
    2. 활용 : 문서요약, 콘텐츠생성, 질문에 대한 답변 생성
  - Chat Model : 사용자와의 상호작용을 통한 연속적인 대화 관리에 적합
    1. 기능 : 메시지의 리스트를 입력으로 받고, 하나의 메시지를 반환한다.
    2. 활용 : 대화형 상황에 최적화, 사용자와의 연속적인 대화를 처리하는 데 사용

LangChain의 LLM 모델 파라미터
  - Temperature : 생성된 텍스트의 다양성을 조정한다. 값이 작으면 예측 가능하고 일관된 출력을 생성, 값이 크면 다양하고 예측하기 어려운 출력을 생성한다
  - Max Tokens(최대 토큰수) : 생성할 최대 토큰 수를 지정한다. 생성할 텍스트의 길이 제한
  - Top P(Top Probability) : 생성과정에서 특정 확률 분포내에서 상위 P% 토큰만을 고려하는 방식으로 출력의 다양성을 조정한다
  - Frequency Penalty(빈도 패널티) : 값이 클수록 이미 등장한 단어나 구절이 다시 등장할 확률을 감소시킨다. 반복을 줄이고 텍스트의 다양성을 증가시킬 수 있다(0 ~ 1)
  - Presnece Penalty(존재 패널티) : 텍스트 내에서 단어의 존재 유무에 그 단어의 선택확률을 조정한다. 값리 클수록 아직 텍스트에 등장하지 않은 새로운 단어의 사용이 장려된다(0 ~ 1)
  - Stop Sequences(정지 시퀀스) : 특정 단어나 구절이 등장할 경우 생성을 멈추도록 설정한다. 출력을 특정 포인트에서 종료하고자 할 때 사용한

In [26]:
# 직접 제어하는 방식은 매우 실무적인 접근
# LLM 모델에 직접 파라미터 전달 예제
# -> 모델의 기본 파라미터와 선택 파라미터를 정의하고,
# -> 모델 인스턴스를 생성할 때 초기값으로 설정

from langchain_openai import ChatOpenAI


# ChatOpenAI 모델 인스턴스를 생성할 때 설정된 파라미터와 추가 키워드 인수를 함께 전달
# 최신 버전에서는 아래와 같이 직관적으로 전달 가능합니다.
model = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature= 0.7,
    max_tokens= 100,
    frequency_penalty= 0.5,
    presence_penalty= 0.5
)


# 모델 호출을 위한 질문 정의
question = '태양계에서 가장 큰 행성은 무엇이니?'
# 모델 호출 및 응답 생성
response = model.invoke(input=question)

# 응답의 내용(텍스트) 출력
response.content

'태양계에서 가장 큰 행성은 목성(Jupiter)입니다. 목성은 지름이 약 139,820킬로미터로, 태양계의 다른 모든 행성을 합친 것보다도 크게 알려져 있습니다. 또한 목성은 가스 거인으로 분류되며, 두꺼운 대기와 강력한 자기장을 가지고 있습니다.'

문제) 다음의 조건을 보고 리팩토링 하시오.

- 리팩토링 핵심 포인트
1. 언패킹(params): 딕셔너리(params)에 담긴 설정값들을 함수 인자로 간편하게 전달합니다. 코드가 훨씬 깔끔해지며, 설정을 별도의 설정 파일(YAML, JSON)에서 불러와 적용할 때 매우 유리합니다.

2. model_kwargs의 활용: LangChain에서 공식적으로 지원하지 않는 최신 API 파라미터(예: frequency_penalty, presence_penalty 등)를 API로 직접 전달할 때 사용하는 표준적인 통로입니다.

- 제어의 효율성:

1. stop=['\n']을 설정. 이는 AI가 답변을 하다가 줄바꿈을 시도할 때 즉시 중단하게 하여, 불필요한 답변 길이를 줄이거나 데이터의 규격(JSON 등)을 지킬 때 매우 효과적인 기법입니다.

In [29]:
from langchain_openai import ChatOpenAI

# ---------------- 개선 코드 ----------------

# 1. 설정값들을 별도의 딕셔너리로 관리 (설정 파일(YAML/JSON)에서 불러오기 최적화)
llm_params = {
    "model" : "gpt-4o-mini",
    "temperature" : 0.7,
    "max_tokens" : 100,
}


# 2. 모델 고유 설정 외에 API로 직접 전달할 옵션은 model_kwargs에 배치
# stop=["\n"] 설정을 추가하여 답변이 줄바꿈을 만나면 즉시 중단하도록 제어
model_kwargs = {
    "frequency_penalty" : 0.5,
    "presence_penalty" : 0.5,
    "stop" : ["\n"]

}



# 3. 딕셔너리 언패킹(**)을 사용하여 모델 인스턴스 생성
# 구조가 깔끔해지고, 설정값을 외부에서 주입받기 매우 편리함
model = ChatOpenAI(
    **llm_params,
    model_kwargs= model_kwargs
)



# 4. 모델 호출
question = '태양계에서 가장 큰 행성은 무엇이니?'
response = model.invoke(input=question)

# 결과 출력
print(f"답변: {response.content}")

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: UserWarning: Parameters {'presence_penalty', 'stop', 'frequency_penalty'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if (await self.run_code(code, result,  async_=asy)):


답변: 태양계에서 가장 큰 행성은 목성(Jupiter)입니다. 목성은 지구의 약 11배에 해당하는 직경을 가지고 있으며, 그 크기와 질량으로 인해 태양계에서 가장 거대한 행성이 됩니다.


LLM 모델 파라미터를 추가로 바인딩
  - bind() 사용의 장점
    1. 모델 설정을 기본값으로 사용하고자할 때 유용
    2. 특수한 상황에서 일부 파라미터를 다르게 적용하고 싶을 때 사용

In [31]:
# LLM 모델 파라미터를 추가로 바인딩 (bind()) 예제
# bind() 사용의 장점:
# 1. 모델 설정을 기본값으로 사용하고자 할 때 유용
# 2. 특수한 상황에서 일부 파라미터를 다르게 적용하고 싶을 때 사용

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# ChatPromptTemplate을 사용하여 시스템 메시지와 사용자 입력 메시지를 포함하는 프롬프트 생성
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '너는 인공지능 질문에 댑변을 할 수 있어.'),
        ('user', '{user_input}'),
    ]
)

# ChatOpenAI 모델 초기화 (기본 max_tokens 설정)
model = ChatOpenAI(model = "gpt-4o-mini", max_tokens= 100)


# 프롬프트에 사용자 입력을 포맷하여 메시지 생성
message = prompt.format_messages(user_input = '인공지능이 가장 발전한 나라와 대표 기술은 뭐지?')

# binding 이전에 모델 직접 호출하여 응답 생성 및 출력
before_answer = model.invoke(message)


# binding 이전 출력
print(before_answer)

# 모델 호출 시 추가적인 인수를 전달하기 위해 bind() 사용
# -> 여기서는 응답의 최대 길이를 50토큰으로 제한
chain = prompt | model.bind(max_tokens=50)


# binding 이후 체인 호출하여 응답 생성 및 출력
after_answer = chain.invoke({"user_input" : "인공지능이 가장 발전한 나라롸 대표 기술은?"})


# binding 이후 출력
print(after_answer)

content='2023년 기준으로 인공지능이 가장 발전한 나라는 일반적으로 미국과 중국으로 꼽힙니다. 이 두 나라는 인공지능 연구 및 개발에 막대한 자원을 투자하고 있으며, 다양한 산업 분야에서 혁신을 이루고 있습니다.\n\n1. **미국**:\n   - **대표 기술**: 자연어 처리(NLP), 자율주행차, 이미지 인식 및 생성 등.\n   - 주요 기업: 구글, 아마존' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 42, 'total_tokens': 142, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_795bd00303', 'id': 'chatcmpl-DyzTtTUPWFiVhDsJETfUgqAc8DeNm', 'service_tier': 'default', 'finish_reason': 'length', 'logprobs': None} id='lc_run--019f3c9d-8b41-70e3-a055-309262f34a4e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 100, 'total_tokens': 142, 'input_token_details': {'audio': 0, 'cache_read':